In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/me-4127-e-project-2/sample_submission.csv
/kaggle/input/me-4127-e-project-2/train.csv
/kaggle/input/me-4127-e-project-2/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import log_loss
import optuna

print("Starting model training process...")

# Load and preprocess data
train = pd.read_csv('/kaggle/input/me-4127-e-project-2/train.csv')
test = pd.read_csv('/kaggle/input/me-4127-e-project-2/test.csv')

# Encode target variable
y = train['Status']
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
test_ids = test['id'].copy()

# Drop unnecessary columns
X = train.drop(['id', 'Status'], axis=1)
test = test.drop(['id'], axis=1)

# Select numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

print(f"\nFeature composition:")
print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

# Enhanced preprocessing pipelines
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# Implement k-fold cross-validation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def create_nn_model(input_dim):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.1),
        Dense(3, activation='softmax')
    ])
    
    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy')
    return model

print("\nOptimizing XGBoost parameters using Optuna...")

def objective(trial):
    xgb_params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'gamma': trial.suggest_float('gamma', 0, 1)
    }
    
    scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]
        
        X_train_processed = preprocessor.fit_transform(X_train_fold)
        X_val_processed = preprocessor.transform(X_val_fold)
        
        model = XGBClassifier(**xgb_params, random_state=42)
        model.fit(X_train_processed, y_train_fold,
                 eval_set=[(X_val_processed, y_val_fold)],
                 early_stopping_rounds=20,
                 verbose=False)
        
        preds = model.predict_proba(X_val_processed)
        fold_score = log_loss(y_val_fold, preds)
        scores.append(fold_score)
    
    return np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
best_params = study.best_params

print(f"\nBest XGBoost parameters:")
for param, value in best_params.items():
    print(f"{param}: {value}")

# Process full dataset
print("\nProcessing full dataset...")
X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(test)

# Initialize arrays for predictions
xgb_models = []
nn_models = []
oof_preds_xgb = np.zeros((len(X), 3))
oof_preds_nn = np.zeros((len(X), 3))

print("\nTraining models with cross-validation:")
fold_scores_xgb = []
fold_scores_nn = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    print(f"\nFold {fold + 1}/{n_splits}")
    
    # Split data
    X_train_fold = X_processed[train_idx]
    X_val_fold = X_processed[val_idx]
    y_train_fold = y[train_idx]
    y_val_fold = y[val_idx]
    
    # Train XGBoost
    print("Training XGBoost...")
    xgb_model = XGBClassifier(**best_params, random_state=42)
    xgb_model.fit(X_train_fold, y_train_fold)
    xgb_models.append(xgb_model)
    
    # Get XGBoost predictions
    xgb_fold_preds = xgb_model.predict_proba(X_val_fold)
    oof_preds_xgb[val_idx] = xgb_fold_preds
    fold_xgb_loss = log_loss(y_val_fold, xgb_fold_preds)
    fold_scores_xgb.append(fold_xgb_loss)
    print(f"XGBoost fold {fold + 1} log loss: {fold_xgb_loss:.4f}")
    
    # Train Neural Network
    print("Training Neural Network...")
    nn_model = create_nn_model(X_processed.shape[1])
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=7, min_lr=1e-6)
    ]
    
    history = nn_model.fit(
        X_train_fold, to_categorical(y_train_fold),
        validation_data=(X_val_fold, to_categorical(y_val_fold)),
        epochs=150,
        batch_size=32,
        callbacks=callbacks,
        verbose=0
    )
    
    nn_models.append(nn_model)
    nn_fold_preds = nn_model.predict(X_val_fold)
    oof_preds_nn[val_idx] = nn_fold_preds
    fold_nn_loss = log_loss(y_val_fold, nn_fold_preds)
    fold_scores_nn.append(fold_nn_loss)
    print(f"Neural Network fold {fold + 1} log loss: {fold_nn_loss:.4f}")

print("\nModel Performance Summary:")
print(f"XGBoost - Average log loss: {np.mean(fold_scores_xgb):.4f} ± {np.std(fold_scores_xgb):.4f}")
print(f"Neural Network - Average log loss: {np.mean(fold_scores_nn):.4f} ± {np.std(fold_scores_nn):.4f}")

# Find optimal ensemble weights
def find_optimal_weights(preds1, preds2, y_true, weight_range=np.arange(0, 1.1, 0.1)):
    best_loss = float('inf')
    best_weight = 0
    
    for w1 in weight_range:
        w2 = 1 - w1
        blend = w1 * preds1 + w2 * preds2
        loss = log_loss(y_true, blend)
        
        if loss < best_loss:
            best_loss = loss
            best_weight = w1
            
    return best_weight, 1 - best_weight, best_loss

# Get optimal weights and ensemble score
w1, w2, ensemble_loss = find_optimal_weights(oof_preds_xgb, oof_preds_nn, y)
print(f"\nEnsemble Results:")
print(f"Optimal weights: XGBoost = {w1:.2f}, NN = {w2:.2f}")
print(f"Ensemble validation log loss: {ensemble_loss:.4f}")

# Make final predictions
print("\nMaking final predictions...")
xgb_preds_test = np.mean([model.predict_proba(X_test_processed) for model in xgb_models], axis=0)
nn_preds_test = np.mean([model.predict(X_test_processed) for model in nn_models], axis=0)

# Combine predictions using optimal weights
final_preds = w1 * xgb_preds_test + w2 * nn_preds_test
final_preds = np.clip(final_preds, 1e-15, 1-1e-15)
final_preds = final_preds / final_preds.sum(axis=1, keepdims=True)

# Create submission file
pred_df = pd.DataFrame(final_preds, columns=['Status_C', 'Status_CL', 'Status_D'])
pred_df.insert(0, 'id', test_ids)
pred_df.to_csv('submission.csv', index=False)

print("\nSubmission file created!")

[I 2025-05-01 18:34:12,152] A new study created in memory with name: no-name-216954fd-5d11-4297-97c3-674372715ed9


Starting model training process...

Feature composition:
Numerical columns: 12
Categorical columns: 6

Optimizing XGBoost parameters using Optuna...


/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds


Best XGBoost parameters:
n_estimators: 500
max_depth: 5
learning_rate: 0.049668428250034015
subsample: 0.8353225754126727
colsample_bytree: 0.6010771364511357
min_child_weight: 6
gamma: 0.2520205354084556

Processing full dataset...

Training models with cross-validation:

Fold 1/5
Training XGBoost...
XGBoost fold 1 log loss: 0.3760
Training Neural Network...


/opt/conda/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Neural Network fold 3 log loss: 0.4199

Fold 4/5
Training XGBoost...
XGBoost fold 4 log loss: 0.3599
Training Neural Network...


/opt/conda/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Neural Network fold 4 log loss: 0.4215

Fold 5/5
Training XGBoost...
XGBoost fold 5 log loss: 0.3470
Training Neural Network...


/opt/conda/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Neural Network fold 5 log loss: 0.3955

Model Performance Summary:
XGBoost - Average log loss: 0.3590 ± 0.0119
Neural Network - Average log loss: 0.4097 ± 0.0122

Ensemble Results:
Optimal weights: XGBoost = 0.90, NN = 0.10
Ensemble validation log loss: 0.3589

Making final predictions...
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Submission file created!
